In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.animation as animation
import time
# numba: Python 3.13 support added in numba 0.61+. Use pip install numba>=0.61
from numba import njit, prange
import warnings

# NOTE: %matplotlib notebook is deprecated in modern Jupyter.
# Use %matplotlib widget (requires ipympl) for interactive plots, or %matplotlib inline.
# %load_ext snakeviz is removed here; run `snakeviz` from the terminal after profiling.

warnings.filterwarnings('ignore')

plt.rcParams["animation.html"] = "jshtml"

# Set interactive=True only if using %matplotlib widget (ipympl installed)
interactive = False
if interactive:
    # %matplotlib widget   # uncomment if ipympl is installed
    blit = True
else:
    # %matplotlib inline   # uncomment when running in Jupyter
    blit = False

In [ ]:
# step 1, Run this

# Self-defined constants used in the simulation
def gamma_1(gamma, del_1, del_2):
    return gamma - 1j * (del_1 + del_2)

def gamma_2(gamma0, del_2):
    return gamma0 - 1j * del_2

def del_2(nu1, nu2, wsg):
    return nu1 - nu2 - wsg

# Gradient function: 4th-order finite difference for interior, 2nd-order at boundaries
@njit(fastmath=True)
def grad(f, h):
    gradients = np.zeros_like(f)
    end = len(f) - 1

    for i in prange(2, len(f) - 2):
        gradients[i] = (-f[i + 2] + 8 * f[i + 1] - 8 * f[i - 1] + f[i - 2]) / (12 * h)

    gradients[0]     = (-3 * f[0] + 4 * f[1] - f[2]) / (2 * h)
    gradients[1]     = (f[2] - f[0]) / (2 * h)
    gradients[end]   = (3 * f[end] - 4 * f[end - 1] + f[end - 2]) / (2 * h)
    gradients[end-1] = (f[end] - f[end - 2]) / (2 * h)

    return gradients


def Vg(c, Rabi, Ng):
    """Group velocity inside the EIT medium."""
    if Rabi == 0:
        return 0.0 + 0.0j
    else:
        a = (Ng / Rabi) ** 2
        return c / (1 - a)


# FIX: np.conjugte -> np.conjugate; k1 was undefined — added k1 parameter
def X(w, del_2, gamma, gamma0, Ng, Rabi, k1, del_1=0):
    """Derived susceptibility A(w)."""
    gamma1 = gamma - 1j * (del_1 + del_2)
    gamma2 = gamma0 - 1j * del_2
    k2 = 1j * Rabi
    k3 = 1j * np.conjugate(Rabi)   # fixed: was np.conjugte (typo)

    a = (gamma1 - 1j * w) - k2 * k3 / (gamma2 - 1j * w)
    return k1 / a   # fixed: k1 was not defined in original; now an explicit argument


Xs = np.vectorize(X)
Vgs = np.vectorize(Vg)

# Params

In [ ]:
# step 2
c = 0.3                           # speed of light (working in ns regime)
g = 10                            # coupling constant
N = 100

# del_1 = Δ = Δ2 = v2 - w_es
# del_2 = δ = Δ1-Δ2 = v1 - v2 - w_sg

del_1 = 0
del_2 = 0      # 0.01 GHz

decay_e   = 50 / 3
dephase_s = 0.0005
dephase_e = 0

gamma = gamma_eg = 0.5 * (decay_e + dephase_e)
gamma0 = gamma_sg = 0.5 * dephase_s
gamma_es = 0.5 * (decay_e + dephase_e + dephase_s)

gamma1 = gamma_1(gamma, del_1, del_2)
gamma2 = gamma_2(gamma0, del_2)

# Signal

In [ ]:
# step 3

def make_pulse(up_time, down_time, c, sharpness, h):
    """Create a single tanh-shaped pulse (spatial)."""
    spatial_width_up   = up_time   * c.real
    spatial_width_down = down_time * c.real

    z  = np.arange(-spatial_width_down * sharpness, spatial_width_up * sharpness, h * sharpness)
    z2 = -z + (z[0] + z[-1])

    edge1 = (np.tanh(z)  + 1) / 2
    edge2 = (np.tanh(z2) + 1) / 2

    z = z / sharpness
    z = np.concatenate((z, z + z[-1] - z[0] + h))
    pulse = np.concatenate((edge1, edge2))

    return z, pulse


def make_pulses(z, pulse, n, h, zrange):
    """Tile a pulse n times and pad to zrange."""
    diff = z[-1] - z[0] + h
    z1 = z
    p  = pulse

    for _ in np.arange(n - 1):
        z     = np.concatenate((z - diff, z1))
        pulse = np.concatenate((pulse, p))

    left_padding  = np.arange(zrange[0], z[0], h)
    right_padding = np.arange(z[-1] + h, zrange[1], h)

    z     = np.concatenate((left_padding, z, right_padding))
    pulse = np.concatenate((np.zeros_like(left_padding), pulse, np.zeros_like(right_padding)))

    return z, pulse.astype(np.complex128)


def time_pulse(storage_time, start_time, ts, sharpness):
    """Generate the temporal Rabi control pulse profile."""
    offset1             = np.arctanh(0.8) + start_time
    storage_start_time  = offset1 - np.arctanh(-0.8)
    storage_end_time    = storage_start_time + storage_time
    offset2             = storage_end_time - np.arctanh(-0.8)

    edge1 = (np.tanh(-(ts - offset1) * sharpness) + 1) / 2
    edge2 = (np.tanh( (ts - offset2) * sharpness) + 1) / 2
    Rabi  = edge1 + edge2

    return Rabi - np.min(Rabi)

# E pulse

Either Gaussian or square pulses — run one of the two blocks below.

In [ ]:
# Gaussian pulse
# %matplotlib inline   # uncomment in Jupyter

t_FWHM = 2
tau    = t_FWHM / 2.355
sigma  = tau * c
FWHM   = sigma * 2.355
sigma  = 2.355 * FWHM
lim    = sigma * 4

pulse_width_outside = 2 * lim
medium_length = 2
enter = lim
exit_ = enter + medium_length   # renamed: 'exit' is a Python builtin

dz = h = 0.004
# FIX: np.arange produces real values; cast to complex128 for downstream numba kernels
z_real = np.arange(-lim, medium_length + lim * 4, h)
z      = z_real.astype(np.complex128)

signal = (10 / (sigma * (2 * np.pi) ** 0.5)) * np.exp((-z_real ** 2) / (2 * sigma ** 2))
signal = signal.astype(np.complex128)

plt.close()
# FIX: plot real parts — matplotlib cannot plot complex arrays directly
plt.plot(z_real, signal.real, label='Gaussian pulse')
plt.xlabel('z')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

In [ ]:
# Square pulse (overrides signal from Gaussian block above)
signal = np.zeros_like(signal)
signal[np.argmin(np.abs(z + 2)) : np.argmin(np.abs(z - 2))] = 1

plt.plot(z.real, signal.real)
plt.xlabel('z')
plt.ylabel('Amplitude')
plt.title('Square pulse')
plt.show()

# Rabi Control

In [ ]:
max_amp_rabi = 30   # in GHz; pulse width inside medium should be < half the medium length

Ng = 1j * g * (N ** 0.5) * np.ones(len(z), dtype=np.complex128)
vg = Vg(c, max_amp_rabi, 1j * g * (N ** 0.5))

pulse_width_inside = vg * (pulse_width_outside / c)
start_time = (pulse_width_outside / c) + (medium_length / 2 - pulse_width_inside) / vg

print(f'Pulse width inside  = {pulse_width_inside.real}')
print(f'Medium length       = {medium_length}')

In [ ]:
sharpness  = 0.7                                   # vary 0 to 1
sharpness  = 10 ** (-1 / (sharpness + 0.3))
extra_time = medium_length / (2 * vg)
storage_time = 100

enter_idx = np.argmin(np.abs(z - enter))           # index where medium starts
exit_idx  = np.argmin(np.abs(z - exit_))           # index where medium ends
Ng[:enter_idx] = 0                                 # zero outside medium
Ng[exit_idx:]  = 0

run_time = (1.5 * storage_time + start_time + extra_time).real
dt = delt = 0.005
tlist = np.arange(0, run_time, delt)

rabi_profile = max_amp_rabi * time_pulse(storage_time, start_time, tlist, sharpness)

plt.close()
plt.plot(tlist, rabi_profile)
plt.xlabel('time')
plt.ylabel('Rabi amplitude')
plt.title('Control pulse profile')
plt.show()

In [ ]:
plt.plot(z.real, Ng.imag)
plt.xlabel('z')
plt.ylabel('Ng (imag)')
plt.title('Medium profile')
plt.show()

In [ ]:
# Stability check: dz/dt >> 1 ensures numerical stability
print(f'dt={dt}, dz={dz}, dz/dt={dz/dt:.4f}, medium_length/dz={medium_length/dz:.0f}')

##### `print(run_time*scale, start_time.real*scale, storage_time*scale, run_time/start_time)`

In [ ]:
print(np.min(z).real, enter.real, exit_.real, np.max(z).real)

In [ ]:
# Initial values
E_profile = signal.astype(np.complex128)
P_profile = np.zeros_like(E_profile)
S_profile = np.zeros_like(E_profile)

l = len(signal)
print('Number of time steps  = ', len(np.arange(0, run_time, delt)))
print('Number of z points    = ', l)
print('Good interval value   = ', int(run_time / (l * delt)))

interval = 5       # save state every `interval` iterations
cut = 0.1
size = (int(run_time / (interval * delt)) + 1, len(z))

In [ ]:
# FIX: nopython=True is redundant and deprecated with @njit — removed.
# FIX: parallel=True added where prange is used so numba enables parallel loops.

@njit(fastmath=True, parallel=True)
def grad_jit(f, h):
    """4th-order centred finite difference gradient (numba-JIT compiled)."""
    gradients = np.zeros_like(f)
    end = len(f) - 1
    for i in prange(2, len(f) - 2):
        gradients[i] = (-f[i+2] + 8*f[i+1] - 8*f[i-1] + f[i-2]) / (12 * h)
    gradients[0]     = (-3*f[0]   + 4*f[1]   - f[2])   / (2 * h)
    gradients[1]     = (f[2]      - f[0])               / (2 * h)
    gradients[end]   = (3*f[end]  - 4*f[end-1] + f[end-2]) / (2 * h)
    gradients[end-1] = (f[end]    - f[end-2])           / (2 * h)
    return gradients


@njit(fastmath=True)
def f_E(E, h, P, k, Ng, vg, dz):
    e_z = grad_jit(E + h, dz)
    return Ng * (P + k) - vg * e_z

@njit(fastmath=True)
def f_P(E, h, P, k, S, l, gamma1, Ng, Rabi):
    return -gamma1 * (P + k) + Ng * (E + h) + 1j * Rabi * (S + l)

@njit(fastmath=True)
def f_S(P, k, S, l, gamma2, Rabi):
    return -gamma2 * (S + l) + 1j * np.conjugate(Rabi) * (P + k)


@njit(fastmath=True)
def solve_rk(E, P, S, Ng, vg, gamma1, gamma2, Rabi, dt, dz, use_rk4=True):
    """Runge-Kutta integrator (RK4 default, RK2 optional)."""
    ek1 = f_E(E, 0, P, 0, Ng, vg, dz)
    pk1 = f_P(E, 0, P, 0, S, 0, gamma1, Ng, Rabi)
    sk1 = f_S(P, 0, S, 0, gamma2, Rabi)

    ek2 = f_E(E, (dt/2)*ek1, P, (dt/2)*pk1, Ng, vg, dz)
    pk2 = f_P(E, (dt/2)*ek1, P, (dt/2)*pk1, S, (dt/2)*sk1, gamma1, Ng, Rabi)
    sk2 = f_S(P, (dt/2)*pk1, S, (dt/2)*sk1, gamma2, Rabi)

    if use_rk4:
        ek3 = f_E(E, (dt/2)*ek2, P, (dt/2)*pk2, Ng, vg, dz)
        pk3 = f_P(E, (dt/2)*ek2, P, (dt/2)*pk2, S, (dt/2)*sk2, gamma1, Ng, Rabi)
        sk3 = f_S(P, (dt/2)*pk2, S, (dt/2)*sk2, gamma2, Rabi)

        ek4 = f_E(E, dt*ek3, P, dt*pk3, Ng, vg, dz)
        pk4 = f_P(E, dt*ek3, P, dt*pk3, S, dt*sk3, gamma1, Ng, Rabi)
        sk4 = f_S(P, dt*pk3, S, dt*sk3, gamma2, Rabi)

        delta_E = (dt/6) * (ek1 + 2*ek2 + 2*ek3 + ek4)
        delta_P = (dt/6) * (pk1 + 2*pk2 + 2*pk3 + pk4)
        delta_S = (dt/6) * (sk1 + 2*sk2 + 2*sk3 + sk4)
    else:  # RK2
        delta_E = dt * ek2
        delta_P = dt * pk2
        delta_S = dt * sk2

    return E + delta_E, P + delta_P, S + delta_S

# Plots

In [ ]:
# FIX: @njit does not support string comparison on keyword args passed at call time.
# The 'method' string argument has been replaced by a bool flag `use_rk4`
# which numba can handle natively.
# FIX: `lim` and `cut` are now explicit parameters instead of captured globals.

@njit(fastmath=True)
def data(E, P, S, z, delt, Ng, rabi_profile, gamma1, gamma2,
         enter_idx, exit_idx, run_time, c, size, dz, lim, cut, interval):

    n_z = len(z)
    Es  = np.zeros(size, dtype=np.complex128)
    Ps  = np.zeros(size, dtype=np.complex128)
    Ss  = np.zeros(size, dtype=np.complex128)
    Zs  = np.zeros(size, dtype=np.complex128)

    vg    = c
    ts    = np.zeros(size[0])
    count = 0
    idx   = 0
    flag  = 0   # 0 = simple cut, 1 = dynamic cut

    n_steps = int(run_time / delt)
    for step in range(n_steps):
        i    = step * delt
        Rabi = rabi_profile[count]
        E, P, S = solve_rk(E, P, S, Ng, vg, gamma1, gamma2, Rabi, delt, dz, True)

        cut_at  = z[np.argmax(E.real)].real - 2 * lim
        cut_idx = np.argmin(np.abs(z.real - cut_at))

        if flag == 1:
            E[:cut_idx] = np.zeros_like(E[:cut_idx])
        else:
            E[:int(n_z * cut)] = np.zeros_like(E[:int(n_z * cut)])

        if count % interval == 0:
            Es[idx] = E.real
            Ps[idx] = P.real
            Ss[idx] = S.real
            Zs[idx] = z.real
            ts[idx] = i
            idx += 1

        count += 1

    return Es, Ps, Ss, Zs, ts

In [ ]:
t0 = time.time()
Es, Ps, Ss, Zs, ts = data(
    E_profile, P_profile, S_profile, z, delt, Ng,
    rabi_profile, gamma1, gamma2,
    enter_idx, exit_idx, run_time, c, size, dz,
    lim, cut, interval   # now explicit args instead of globals
)
print(f'{time.time() - t0:.3f} s taken to run')

filt = ~(Es == 0).all(axis=1)
Es = Es[filt]
Ps = Ps[filt]
Ss = Ss[filt]
Zs = Zs[filt]
ts = ts[filt]
print('Output shape:', Es.shape)

#### The matrices `Es`, `Ps`, `Ss` hold the electric field, polarization, and spin wave
at all discretized spatial points across sampled time steps.

In [ ]:
# Benchmark notes (original → updated):
# Before optimization:  ~53 s  (3 186 000 iterations, 707-point z)
# After njit + RK4:     ~0.33 s

In [ ]:
zidx = np.argmax(Es[int(np.argmin(rabi_profile) / interval), :])
arr  = Es[:, zidx]

storage_start_at = ts[np.argmax(arr)]
storage_ends_at  = storage_start_at + storage_time
idx_end          = np.argmin(np.abs(ts - storage_ends_at))
arr_slice        = arr[np.argmax(arr) : idx_end]

decay_time  = ts[np.argmin(np.abs(arr_slice - (1 / np.e)))] - ts[np.argmax(arr)]
entry_time  = ts[np.argmax(Es[:, enter_idx])]
exit_time   = ts[np.argmax(Es[:, exit_idx])]
time_delay  = exit_time - entry_time - (exit_ - enter) / c   # uses exit_ (not builtin exit)

peak_amp_at_t = np.max(Es, axis=1)
min_peak      = np.min(peak_amp_at_t).real
optical_depth = (-(Ng.max() ** 2) * (exit_ - enter) / (c * gamma)).real

print(f'Entry Time            = {entry_time:.3e} s')
print(f'Exit Time             = {exit_time:.3e} s')
print(f'Time delay            = {time_delay:.3e} s')
print(f'Minimum peak value    = {min_peak:.3g}')
print(f'Optical Depth         = {optical_depth:.3g} m')
print(f'Medium Length         = {(exit_ - enter):.3g} m')

# Static Time Plots

In [ ]:
# Decay of peak amplitude with time
plt.close()
plt.plot(ts, peak_amp_at_t)
plt.xlabel('time')
plt.ylabel('Peak amplitude')
plt.title('Peak amplitude decay')
plt.show()

In [ ]:
z1, z2, z3 = enter_idx - 2, (enter_idx + exit_idx) // 2, exit_idx

plt.close()
plt.style.use('ggplot')
fig, axs = plt.subplots(1, 3, figsize=(24, 8))

axs[0].plot(ts, Es[:, z1] ** 2, linewidth=1, label='Input')
axs[0].plot(ts, Es[:, z2] ** 2, linewidth=1, label='At midpoint')
axs[0].plot(ts, Es[:, z3] ** 2, linewidth=1, label='Output')
ax1 = axs[0].twinx()
ax1.plot(np.arange(0, run_time, delt), rabi_profile, linewidth=0.5, color='black', linestyle='--')
axs[0].set_title('E')

axs[1].plot(ts, Ps[:, z1] ** 2, linewidth=1)
axs[1].plot(ts, Ps[:, z2] ** 2, linewidth=1)
axs[1].plot(ts, Ps[:, z3] ** 2, linewidth=1)
ax2 = axs[1].twinx()
ax2.plot(np.arange(0, run_time, delt), rabi_profile, linewidth=0.5, color='black', linestyle='--')
axs[1].set_title('P')

axs[2].plot(ts, Ss[:, z1] ** 2, linewidth=1)
axs[2].plot(ts, Ss[:, z2] ** 2, linewidth=1)
axs[2].plot(ts, Ss[:, z3] ** 2, linewidth=1)
ax3 = axs[2].twinx()
ax3.plot(np.arange(0, run_time, delt), rabi_profile, linewidth=0.5, color='black', linestyle='--')
axs[2].set_title('S')

axs[0].legend()
plt.tight_layout()
plt.show()

# 3D Plots

In [ ]:
(m, n) = Zs.shape
t_grid = np.tile(ts, (n, 1)).T    # FIX: np.tile is cleaner than np.array([ts]*n).T

fig = plt.figure()
ax  = fig.add_subplot(111, projection='3d')   # FIX: plt.axes(projection='3d') deprecated
ax.plot_surface(t_grid, Zs, Es)
ax.set_title('Surface plot')
ax.set_xlabel('time')
ax.set_ylabel('z')
ax.set_zlabel('Amplitude')
plt.show()

# Animation

In [ ]:
# Animation of signal field propagation
ys = Es
animation_speed = 2

plt.close()
fig = plt.figure()
ax  = fig.add_subplot(
    111,
    autoscale_on=False,
    xlim=[np.min(z.real), np.max(z.real)],   # FIX: real parts for axis limits
    ylim=[np.min(ys), np.max(ys)]
)
ax.grid()
ax.set_xlabel('z-axis')
ax.axvline(enter.real, color='crimson')   # FIX: real parts for axvline
ax.axvline(exit_.real, color='crimson')

line, = ax.plot([], [], lw=1.5)

def init():
    line.set_data([], [])
    return (line,)

def animate(i):
    line.set_data(Zs[animation_speed * i], ys[animation_speed * i])
    return (line,)

anim = FuncAnimation(
    fig, animate, init_func=init,
    frames=len(ys) // animation_speed,
    interval=1, blit=False
)
anim

In [ ]:
import matplotlib.pyplot as plt

# Replace the string below with the path you got from 'where.exe ffmpeg'
plt.rcParams['animation.ffmpeg_path'] = r'C:\Users\RikteemBhowmick\AppData\Local\Microsoft\WinGet\Links\ffmpeg.exe'

def make_animation(output_file, na_array, nb_array, nc_array, anim_speed):
    """Save an MP4 animation of E, S, P fields using ffmpeg."""
    fps = 10
    FFMpegWriter = animation.FFMpegWriter(fps=fps,   # FIX: use FFMpegWriter directly
                                          metadata=dict(title='EIT', artist='Matplotlib'))
    fig, ax = plt.subplots()
    plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    plt.title('EIT')

    xlimit = len(na_array[0])
    ylimit1 = np.min(nb_array)
    ylimit2 = np.max(na_array)

    with FFMpegWriter.saving(fig, output_file, 100):
        for i in range(len(na_array) // anim_speed):   # FIX: iterate over na_array not Es
            ax.set_xlim([0, xlimit])
            ax.set_ylim([ylimit1, ylimit2])
            ax.plot(na_array[i], color='blue',  label='Signal Field')
            ax.plot(nb_array[i], color='red',   label='Spin Field')
            ax.plot(nc_array[i], color='black', label='Polarization')
            if i == 0:
                ax.legend()                            # FIX: only add legend once
            FFMpegWriter.grab_frame()
            ax.clear()

    plt.clf()

In [ ]:
Ps0 = Ps / np.max(np.abs(Ps))
Es0 = (Es / np.max(np.abs(Es))) ** 2
Ss0 = Ss / np.max(np.abs(Ss))

anim_speed = 5
n_frames   = len(ts) // anim_speed

# FIX: use array slicing with stride instead of a Python loop for efficiency
E_anim = np.asarray([Es0[i * anim_speed] for i in range(n_frames)])
P_anim = np.asarray([Ps0[i * anim_speed] for i in range(n_frames)])
S_anim = np.asarray([Ss0[i * anim_speed] for i in range(n_frames)])

make_animation('eit_animate_testnew.mp4', E_anim, S_anim, P_anim, anim_speed)

In [ ]:
print('S_anim min:', S_anim.min())

In [ ]:
np.save('100nsEs.npy', Es)
np.save('100nsPs.npy', Ps)
np.save('100nsSs.npy', Ss)
print('Arrays saved.')